# Traveler Insights · Del TF-IDF al Transformer con CRISP-ML(Q)
**Clasificación de sentimiento de reseñas turísticas: tres generaciones de modelos, evaluación rigurosa y despliegue**

Notebook **independiente**: no usa la API de Kaggle ni módulos externos al notebook. Basta con tener `train.csv` (y opcionalmente `test.csv`) de la competencia *Traveler Insights UPTC 2026*.

| Modelo | Representación | Qué conserva del texto | Qué pierde |
|---|---|---|---|
| **A · Base** | TF-IDF de palabras y caracteres + **rasgos léxicos** del curso | Palabras y fragmentos frecuentes; pronombres, negación, adjetivos | Orden y significado: "no es bueno" ≈ "es bueno, no" |
| **B · Word embeddings** | Vectores densos FastText (subpalabras) promediados + rasgos léxicos | Similitud semántica: "sucio" ≈ "mugriento" | Orden y contexto: un solo vector por palabra, sin importar la frase |
| **C · Transformer** | BETO (BERT en español) afinado | Contexto bidireccional: "no estuvo nada mal" | Interpretabilidad, costo bajo, rapidez |

### Fases de CRISP-ML(Q) en este notebook
| Fase | Sección |
|---|---|
| 1. Entendimiento del negocio y de los datos | §1 |
| 2. Preparación de datos (con requisitos de calidad) | §2 |
| 3. Modelado (A, B, C con el mismo protocolo) | §3 |
| 4. Evaluación (comparación justa y compuerta de calidad) | §4 |
| 5. Despliegue (API, app web y opciones de publicación) | §5 |
| 6. Monitoreo y mantenimiento | §6 |

### Cómo obtener los datos (sin API)
1. Entre a la competencia en Kaggle, pestaña **Data**, y pulse **Download All**.
2. Descomprima y deje `train.csv` (y `test.csv`) junto a este notebook, o ajuste las rutas en §0.
3. En Google Colab, ejecute la celda opcional de §0 para subir los archivos.

### Requisitos
```
pip install pandas numpy scikit-learn scipy matplotlib joblib gensim torch transformers fastapi httpx
```
El modelo C necesita GPU para entrenar en minutos (Colab: *Entorno de ejecución → Cambiar tipo → GPU*). En CPU funciona, pero es lento.

## 0. Configuración

In [ ]:
# ====== CONFIGURACIÓN (edite solo esta celda) ======
DATA_PATH  = "train.csv"          # datos etiquetados de la competencia
TEST_PATH  = "test.csv"           # opcional: sin etiquetas, para monitoreo de deriva (§6)
TEXT_COL   = "Comentario"
LABEL_COL  = "Sentimiento"
SEED       = 42
TEST_SIZE  = 0.20                 # partición de prueba: se separa al inicio y se usa una sola vez
CV_FOLDS   = 5

RUN_EMBEDDINGS  = True
EMB_PRETRAINED  = None            # opcional: ruta a vectores preentrenados (.vec/.bin), p. ej. fastText cc.es.300
RUN_TRANSFORMER = True
TRANSFORMER_MODEL = "dccuchile/bert-base-spanish-wwm-cased"   # BETO (Cañete et al., 2020)
TR_EPOCHS, TR_LR, TR_BATCH, TR_MAXLEN = 4, 3e-5, 16, 128

DEPLOY_DIR = "deploy"             # carpeta donde se exporta lo necesario para desplegar

In [ ]:
# Opcional (Google Colab): subir train.csv y test.csv desde su computador
try:
    from google.colab import files  # noqa: F401
    EN_COLAB = True
except ImportError:
    EN_COLAB = False
if EN_COLAB:
    import os
    if not os.path.exists(DATA_PATH):
        print("Seleccione train.csv (y test.csv si lo tiene):")
        files.upload()

In [ ]:
import json, math, re, time, unicodedata, warnings
from collections import Counter
from pathlib import Path

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy.stats import binomtest
from sklearn.base import BaseEstimator, TransformerMixin, clone
from sklearn.dummy import DummyClassifier
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (ConfusionMatrixDisplay, accuracy_score, classification_report, f1_score,
                             recall_score)
from sklearn.model_selection import StratifiedKFold, train_test_split
from sklearn.pipeline import FeatureUnion, Pipeline
from sklearn.preprocessing import StandardScaler

warnings.filterwarnings("ignore")
pd.set_option("display.max_colwidth", 160)
plt.rcParams["figure.dpi"] = 110
ROJO, GRIS, OSCURO, AZUL = "#C10230", "#6F7271", "#222222", "#2E6FD8"
rng = np.random.default_rng(SEED)

**Módulo propio del notebook.** Las funciones de texto, los rasgos léxicos y el vectorizador de embeddings se escriben en `ti_rasgos.py` con `%%writefile`. Así el modelo entrenado se puede guardar y cargar fuera del notebook, por ejemplo en la API de §5. Se explican en §3.1 y §3.2.

In [ ]:
%%writefile ti_rasgos.py
"""Funciones de texto, rasgos léxicos del curso y embeddings (generado por el notebook Traveler Insights)."""
import re
import unicodedata

import numpy as np
from sklearn.base import BaseEstimator, TransformerMixin

_PAL = re.compile(r"\w+")


def normalizar(texto):
    """Minúsculas, sin tildes y espacios colapsados."""
    t = unicodedata.normalize("NFKD", str(texto).lower())
    t = "".join(ch for ch in t if not unicodedata.combining(ch))
    return re.sub(r"\s+", " ", t).strip()


# Léxico de los modelos anteriores del curso (courseNLP/examples/logic/lexical_features.py)
LEXICO_ES = {'first_person_singular': ['yo', 'mio', 'mi'],
 'second_person_singular': ['tú', 'tu', 'tuyo', 'usted', 'vos'],
 'third_person_singular': ['él', 'el', 'ella', 'ese', 'esa', 'aquel'],
 'first_person_plurar': ['nosotros', 'nuestros'],
 'second_person_plurar': ['tuyos', 'tuyas', 'ustedes', 'vosotros', 'vosotras'],
 'third_person_plurar': ['ellos', 'ellas', 'ese', 'esos', 'aquellas', 'aquellos'],
 'adverb_time': ['ya', 'aún', 'hoy', 'tarde', 'pronto', 'todavía', 'ayer', 'recién', 'nunca', 'siempre', 'jamás',
                 'ahora'],
 'adverb_neg': ['tampoco', 'no', 'en absoluto', 'nunca'],
 'adverb_place': ['ahí', 'allí', 'aquí', 'acá', 'delante', 'detrás', 'arriba', 'abajo', 'cerca', 'lejos', 'encima',
                  'fuera', 'dentro'],
 'adverb_mode': ['mal', 'bien', 'regular', 'despacio', 'así', 'mejor', 'peor', 'claro', 'similar', 'fácilmente'],
 'adverb_cant': ['muy', 'más', 'poco', 'bastante', 'bastante', 'menos', 'mucho', 'algo', 'casi'],
 'adjetives_neg': ['aburrido', 'ácido', 'amargo', 'ancho', 'atrevido', 'bajo', 'blando', 'caliente', 'común',
                   'conocido', 'corto', 'débil', 'delgado', 'diferente', 'difícil', 'duro', 'enfermo', 'estrecho',
                   'falso', 'feo', 'frío', 'húmedo', 'imposible', 'izquierdo', 'lento', 'malo', 'menor', 'muerto',
                   'negro', 'peor', 'pequeño', 'pobre', 'poco', 'popular', 'raro', 'rojo', 'salado', 'simple',
                   'sinvergüenza', 'solo', 'soso', 'tímido', 'tonto', 'triste', 'viejo'],
 'adjetives_pos': ['alegre', 'alto', 'azul', 'blanco', 'bonito', 'buen', 'bueno', 'caliente', 'capaz', 'central',
                   'contento', 'derecho', 'diferente', 'divertido', 'dulce', 'duro', 'exterior', 'famoso', 'final',
                   'fresco', 'fuerte', 'gran', 'grande', 'guapo', 'guay', 'igual', 'imposible', 'interesante',
                   'joven', 'largo', 'listo', 'mayor', 'mejor', 'mucho', 'musical', 'nacional', 'natural', 'nuevo',
                   'perfecto', 'popular', 'posible', 'primer', 'primero', 'principal', 'próximo', 'rápido', 'real',
                   'recto', 'rico', 'salado', 'sano', 'seco', 'útil', 'verdadero', 'verde', 'vivo'],
 'who_general': ['familia', 'familiar', 'familiares', 'acompañante', 'aprendices', 'aprendiz', 'artista', 'asistente',
                 'compinche', 'directiva', 'directivo', 'especialista', 'estudiante', 'estudiantes', 'joven',
                 'jovenes', 'lider', 'pareja', 'projimo'],
 'who_male': ['abuelo', 'alcalde', 'medico', 'alumno', 'alumnos', 'amigo', 'amigos', 'asesor', 'chiquillo', 'compa',
              'compadre', 'compañero', 'compañeros', 'decano', 'director', 'doctor', 'empleado', 'empresario',
              'entrenador', 'esposo', 'esposito', 'experto', 'expertos', 'funcionario', 'hermano', 'ingeniero',
              'investigador', 'jefe', 'jugador', 'padre', 'marido', 'medico', 'muchacho', 'muchachos', 'niño',
              'niños', 'novio', 'hijo', 'primo', 'papa', 'pequeños', 'profesor', 'pequeño', 'secretario', 'socio',
              'tutor', 'papi', 'tio', 'nieto'],
 'who_female': ['alumna', 'alumnas', 'amiga', 'amigas', 'asesora', 'chiquilla', 'comadre', 'compañera', 'compañeras',
                'dama', 'decana', 'directora', 'doctora', 'empleada', 'empresaria', 'esposa', 'esposita', 'experta',
                'expertas', 'funcionaria', 'hermana', 'ingeniera', 'investigadora', 'jefa', 'jugadora', 'madre',
                'mama', 'medica', 'muchacha', 'muchachas', 'mujer', 'niñas', 'novia', 'novicia', 'hija', 'prima',
                'pequeña', 'pequeñas', 'profesora', 'secretaria', 'señora', 'socia', 'tutora', 'mami', 'tia',
                'nieta'],
 'hate': ['animosidad', 'antagonismo', 'enemistad', 'odio', 'horror', 'hostilidad', 'aborrecimiento', 'dolor',
          'rencor', 'resentimiento', 'venganza', 'veneno', 'aborrecimiento', 'abominación ', 'anatema', 'animus',
          'antipatía', 'aversión', 'molestar', 'bugbear', 'detestación ', 'disgusto', 'execración', 'escarcha',
          'agravio', 'queja', 'irritante', 'malevolencia', 'malignidad', 'molestia', 'objeción', 'odio', 'irritación',
          'repugnancia', 'repulsión', 'repulsión', 'desprecio', 'despecho', 'problemas', 'bestia negra',
          'mala voluntad', 'desagradable', 'mirada desagradable', 'ningún amor perdido']}


def _preparar(lex):
    out = {}
    for cat, entradas in lex.items():
        unas, frases = set(), []
        for e in entradas:
            n = normalizar(e)
            if " " in n:
                frases.append(re.compile(r"" + re.escape(n) + r""))
            elif n:
                unas.add(n)
        out[cat] = (unas, frases)
    return out


_LEX = _preparar(LEXICO_ES)
ADVERBIOS = ["adverb_time", "adverb_neg", "adverb_place", "adverb_mode", "adverb_cant"]
NOMBRES_LEX = list(LEXICO_ES) + ["adverb_all", "n_words", "avg_word", "kur_word", "skew_word", "lexical_diversity"]


def rasgos(texto):
    """Vector de rasgos léxicos (orden de NOMBRES_LEX), con las 4 correcciones documentadas."""
    t = normalizar(texto)
    tok = _PAL.findall(t)
    v = {c: float(sum(w in _LEX[c][0] for w in tok) + sum(len(p.findall(t)) for p in _LEX[c][1])) for c in LEXICO_ES}
    v["adverb_all"] = sum(v[a] for a in ADVERBIOS)
    L = np.array([len(w) for w in tok], float)
    m2 = L.var() if len(L) else 0.0
    v["n_words"] = float(len(tok))
    v["avg_word"] = float(L.mean()) if len(L) else 0.0
    v["skew_word"] = float(((L - L.mean()) ** 3).mean() / m2 ** 1.5) if m2 > 0 else 0.0   # con signo
    v["kur_word"] = float(((L - L.mean()) ** 4).mean() / m2 ** 2 - 3) if m2 > 0 else 0.0  # con signo
    v["lexical_diversity"] = len(set(tok)) / len(tok) if tok else 0.0                     # por palabras
    return [v[n] for n in NOMBRES_LEX]


class RasgosLexicos(BaseEstimator, TransformerMixin):
    """Rasgos léxicos como transformador de scikit-learn (no aprende nada en fit)."""

    def fit(self, X, y=None):
        return self

    def transform(self, X):
        return np.array([rasgos(t) for t in X])


class EmbeddingsPromedio(BaseEstimator, TransformerMixin):
    """Promedio de vectores de palabra: FastText entrenado en el fold, o vectores preentrenados."""

    def __init__(self, dim=100, epochs=40, min_count=1, pretrained=None, seed=42):
        self.dim, self.epochs, self.min_count, self.pretrained, self.seed = dim, epochs, min_count, pretrained, seed

    @staticmethod
    def _tok(t):
        return _PAL.findall(normalizar(t))

    def fit(self, X, y=None):
        from gensim.models import FastText, KeyedVectors
        if self.pretrained:
            binario = str(self.pretrained).endswith(".bin")
            self.kv_ = KeyedVectors.load_word2vec_format(self.pretrained, binary=binario, limit=300_000)
        else:   # se entrena SOLO con los textos de entrenamiento del pliegue: sin fuga de información
            frases = [self._tok(t) for t in X]
            self.kv_ = FastText(frases, vector_size=self.dim, window=5, min_count=self.min_count,
                                epochs=self.epochs, seed=self.seed, workers=1, sg=1).wv
        return self

    def transform(self, X):
        out = np.zeros((len(X), self.kv_.vector_size))
        for i, t in enumerate(X):
            # FastText entrenado da vector a cualquier palabra (subpalabras); los preentrenados solo a su vocabulario
            vs = [self.kv_[w] for w in self._tok(t) if not self.pretrained or w in self.kv_.key_to_index]
            if vs:
                out[i] = np.mean(vs, axis=0)
        return out

In [ ]:
import importlib, sys
sys.path.insert(0, ".")
sys.modules.pop("ti_rasgos", None)
from ti_rasgos import (ADVERBIOS, LEXICO_ES, NOMBRES_LEX, EmbeddingsPromedio, RasgosLexicos, normalizar, rasgos)
_PAL = re.compile(r"\w+")
print(f"{len(NOMBRES_LEX)} rasgos léxicos · {sum(len(v) for v in LEXICO_ES.values())} entradas en el léxico")

## 1. Fase 1 · Entendimiento del negocio y de los datos
**Problema:** un operador turístico recibe reseñas en varias plataformas y las lee a mano, tarde y sin agregación.

**Decisión que cambia:** qué reseñas se atienden automáticamente y cuáles revisa una persona, y qué quejas atender primero.

**Tarea de PLN:** clasificar cada reseña como *positivo*, *neutral* o *negativo*.

Los **criterios de aceptación** se fijan *antes* de ver resultados. Cambiarlos después sería mover el arco cuando ya se pateó el balón.

In [ ]:
COMPUERTA = {
    "min_macro_f1": 0.50,        # mínimo aceptable para el negocio
    "beats_dummy_by": 0.10,      # mejora mínima sobre predecir siempre la clase mayoritaria
    "min_recall_negativo": 0.30, # el sistema debe detectar al menos parte de las quejas
}
TAU = 0.70                       # umbral de confianza: por debajo, la reseña va a revisión humana
pd.Series(COMPUERTA, name="criterio")

## 2. Fase 2 · Preparación y validación de datos
Requisitos de calidad: filas suficientes, ejemplos por clase, pocos duplicados y ningún texto con etiquetas contradictorias. Si no se cumplen, **no se entrena**.

In [ ]:
raw = pd.read_csv(DATA_PATH)
print("Columnas:", list(raw.columns))
assert TEXT_COL in raw.columns and LABEL_COL in raw.columns, "Revise TEXT_COL y LABEL_COL en §0"

d = raw[[TEXT_COL, LABEL_COL]].rename(columns={TEXT_COL: "texto", LABEL_COL: "etiqueta"}).dropna()
d["texto"] = d["texto"].astype(str); d["etiqueta"] = d["etiqueta"].astype(str)
d = d[d["texto"].str.strip() != ""]
d["_n"] = d["texto"].map(normalizar)
conflictivos = d.groupby("_n")["etiqueta"].nunique().loc[lambda s: s > 1].index
calidad = {"filas_originales": len(raw), "filas_validas": len(d),
           "tasa_duplicados": round(float(d["_n"].duplicated().mean()), 4),
           "textos_conflictivos": int(len(conflictivos))}
d = d[~d["_n"].isin(conflictivos)].drop_duplicates("_n").drop(columns="_n").reset_index(drop=True)
dist = d["etiqueta"].value_counts()
calidad.update({"filas_finales": len(d), "clases": dist.to_dict(), "razon_desbalance": round(dist.max() / dist.min(), 1)})

REQUISITOS = {"min_filas": len(d) >= 200, "min_por_clase": dist.min() >= 20,
              "duplicados_<=20%": calidad["tasa_duplicados"] <= 0.20, "conflictos_<=2%": len(conflictivos) <= 0.02 * len(d)}
display(pd.Series(calidad, dtype=object).to_frame("valor"))
display(pd.Series(REQUISITOS).map({True: "✔", False: "✘"}).to_frame("requisito"))
assert all(REQUISITOS.values()), "Los datos no cumplen los requisitos de calidad: se detiene el proceso"

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(11, 3.2))
dist.sort_values().plot.barh(ax=ax[0], color=ROJO); ax[0].set_title(f"Clases · desbalance {calidad['razon_desbalance']}")
d["texto"].str.split().str.len().clip(upper=150).plot.hist(bins=40, ax=ax[1], color=GRIS); ax[1].set_title("Palabras por reseña")
plt.tight_layout(); plt.show()

**Partición única y estratificada.** La prueba se separa ahora y no se toca hasta §4. Para el Transformer, además, se separa una validación dentro del entrenamiento para decidir cuándo detenerse.

In [ ]:
X_tr, X_te, y_tr, y_te = train_test_split(d["texto"], d["etiqueta"], test_size=TEST_SIZE,
                                          stratify=d["etiqueta"], random_state=SEED)
CLASES = sorted(d["etiqueta"].unique())
fuga = len(set(X_tr.map(normalizar)) & set(X_te.map(normalizar)))
print(f"Entrenamiento {len(X_tr)} · prueba {len(X_te)} · textos repetidos entre ambos: {fuga}")
print("Prueba por clase:", y_te.value_counts().to_dict())

## 3. Fase 3 · Modelado
Los tres modelos se evalúan con **el mismo protocolo**: validación cruzada estratificada en el entrenamiento y la misma partición de prueba al final. Para tratar el desbalance, todos usan **pesos de clase**.

In [ ]:
RESULTADOS = {}   # nombre → {"proba": matriz de probabilidades en prueba, "clases", "t_train", "t_inf_ms", "mb"}

def tamano_mb(obj):
    import io
    b = io.BytesIO(); joblib.dump(obj, b); return b.tell() / 1e6

def validacion_cruzada(modelo, X, y, k=CV_FOLDS):
    skf = StratifiedKFold(n_splits=k, shuffle=True, random_state=SEED)
    f1s = [f1_score(y.iloc[va], clone(modelo).fit(X.iloc[tr], y.iloc[tr]).predict(X.iloc[va]), average="macro")
           for tr, va in skf.split(X, y)]
    return float(np.mean(f1s)), float(np.std(f1s))

def registrar(nombre, modelo, cv=None):
    t0 = time.perf_counter(); modelo.fit(X_tr, y_tr); t_train = time.perf_counter() - t0
    t0 = time.perf_counter(); proba = modelo.predict_proba(X_te); t_inf = (time.perf_counter() - t0) / len(X_te) * 1000
    RESULTADOS[nombre] = {"proba": proba, "clases": list(modelo.classes_), "t_train": t_train, "t_inf_ms": t_inf,
                          "mb": tamano_mb(modelo), "cv": cv, "modelo": modelo}
    pred = np.asarray(modelo.classes_)[proba.argmax(1)]
    cvtxt = f"CV {cv[0]:.3f} ± {cv[1]:.3f} · " if cv else ""
    print(f"{nombre:32s} {cvtxt}prueba macro-F1 {f1_score(y_te, pred, average='macro'):.3f}")

registrar("Dummy (mayoritaria)", DummyClassifier(strategy="most_frequent"))

### 3.1 Modelo A · TF-IDF + rasgos léxicos (modelo base)
Los rasgos léxicos vienen de `lexical_features.py` de los modelos anteriores del curso (TASS). Se reimplementan con cuatro correcciones:
1. Léxico y texto se normalizan igual, así que las entradas con tilde ("él", "aún", "más") sí se encuentran.
2. Curtosis y asimetría conservan el signo (el original aplicaba `abs()`).
3. La diversidad léxica se calcula por palabras, no por caracteres.
4. Se omiten los rasgos que solo miden longitud (`weighted_position`) y las etiquetas de Twitter.

In [ ]:
ejemplo = "No me gustó: la playa muy sucia y el personal grosero, nunca más vuelvo"
{k: v for k, v in zip(NOMBRES_LEX, rasgos(ejemplo)) if v}

In [ ]:
def modelo_A(lexico=True):
    bloques = [("word", TfidfVectorizer(preprocessor=normalizar, ngram_range=(1, 2), min_df=2, sublinear_tf=True)),
               ("char", TfidfVectorizer(preprocessor=normalizar, analyzer="char_wb", ngram_range=(3, 5), min_df=3, sublinear_tf=True))]
    if lexico:   # el escalador está dentro del pipeline: se ajusta en cada pliegue, sin fuga de información
        bloques.append(("lex", Pipeline([("r", RasgosLexicos()), ("s", StandardScaler())])))
    return Pipeline([("feats", FeatureUnion(bloques)),
                     ("clf", LogisticRegression(C=4.0, class_weight="balanced", max_iter=3000))])

registrar("A0 · TF-IDF (sin léxico)", modelo_A(False), validacion_cruzada(modelo_A(False), X_tr, y_tr))
registrar("A · TF-IDF + léxico", modelo_A(True), validacion_cruzada(modelo_A(True), X_tr, y_tr))

### 3.2 Modelo B · Word embeddings (FastText)
Cada palabra se representa con un vector denso. Por defecto se entrena **FastText** solo con las reseñas de entrenamiento: así no hay fuga de información, y los n-gramas de caracteres dan vectores incluso a palabras no vistas o mal escritas. La reseña se representa con el promedio de los vectores de sus palabras.

Si `EMB_PRETRAINED` apunta a vectores preentrenados en español (por ejemplo, fastText `cc.es.300`), se usan esos. Con solo cientos de reseñas, los vectores preentrenados suelen aportar más semántica.

In [ ]:
def modelo_B(lexico=True):
    bloques = [("emb", Pipeline([("e", EmbeddingsPromedio(pretrained=EMB_PRETRAINED, seed=SEED)), ("s", StandardScaler())]))]
    if lexico:
        bloques.append(("lex", Pipeline([("r", RasgosLexicos()), ("s", StandardScaler())])))
    return Pipeline([("feats", FeatureUnion(bloques)),
                     ("clf", LogisticRegression(C=1.0, class_weight="balanced", max_iter=3000))])

if RUN_EMBEDDINGS:
    registrar("B0 · Embeddings (sin léxico)", modelo_B(False), validacion_cruzada(modelo_B(False), X_tr, y_tr))
    registrar("B · Embeddings + léxico", modelo_B(True), validacion_cruzada(modelo_B(True), X_tr, y_tr))

In [ ]:
if RUN_EMBEDDINGS:   # ¿qué aprendió? vecinos semánticos de algunas palabras
    kv = RESULTADOS["B · Embeddings + léxico"]["modelo"].named_steps["feats"].transformer_list[0][1].named_steps["e"].kv_
    for w in ["sucio", "excelente", "caro", "playa", "atencion"]:
        try: print(f"{w:10s} →", ", ".join(x for x, _ in kv.most_similar(w, topn=5)))
        except KeyError: print(f"{w:10s} → (fuera del vocabulario)")

**Para discutir:** con FastText entrenado en pocas reseñas, los vecinos reflejan **coocurrencia en este corpus** más que sinónimos: "playa" puede quedar cerca de "agua" o "arena". Así funciona la semántica distribucional, y también es su límite cuando hay pocos datos.

### 3.3 Modelo C · Transformer (BETO afinado)
Se afina un BERT preentrenado en español (BETO; Cañete et al., 2020) para clasificar secuencias:
- La representación de cada palabra **depende de toda la reseña**, gracias a la autoatención.
- Se usa **pérdida ponderada por clase**, por el desbalance.
- Se usa **parada temprana** según el macro-F1 en una validación separada del entrenamiento; la prueba no se toca.
- Por costo, no se hace validación cruzada. Es una diferencia de protocolo que hay que declarar al comparar.

In [ ]:
if RUN_TRANSFORMER:
    import torch
    from torch.utils.data import DataLoader, TensorDataset
    from transformers import AutoModelForSequenceClassification, AutoTokenizer

    torch.manual_seed(SEED)
    dev = "cuda" if torch.cuda.is_available() else "cpu"
    lab2id = {c: i for i, c in enumerate(CLASES)}
    Xa, Xv, ya, yv = train_test_split(X_tr, y_tr, test_size=0.15, stratify=y_tr, random_state=SEED)

    tok = AutoTokenizer.from_pretrained(TRANSFORMER_MODEL)
    def cargar(X, y=None, shuffle=False):
        enc = tok(list(X), truncation=True, max_length=TR_MAXLEN, padding=True, return_tensors="pt")
        tens = [enc["input_ids"], enc["attention_mask"]]
        if y is not None: tens.append(torch.tensor([lab2id[v] for v in y]))
        return DataLoader(TensorDataset(*tens), batch_size=TR_BATCH, shuffle=shuffle)

    def probas(modelo, loader):
        modelo.eval(); out = []
        with torch.no_grad():
            for b in loader:
                out.append(torch.softmax(modelo(input_ids=b[0].to(dev), attention_mask=b[1].to(dev)).logits, -1).cpu())
        return torch.cat(out).numpy()

    modelo_C = AutoModelForSequenceClassification.from_pretrained(
        TRANSFORMER_MODEL, num_labels=len(CLASES), id2label=dict(enumerate(CLASES)), label2id=lab2id,
        ignore_mismatched_sizes=True).to(dev)   # la cabeza de clasificación se inicializa para nuestras 3 clases
    conteo = ya.value_counts()
    pesos = torch.tensor([len(ya) / (len(CLASES) * conteo[c]) for c in CLASES], dtype=torch.float).to(dev)
    perdida = torch.nn.CrossEntropyLoss(weight=pesos)
    opt = torch.optim.AdamW(modelo_C.parameters(), lr=TR_LR, weight_decay=0.01)
    dl_tr, dl_va = cargar(Xa, ya, shuffle=True), cargar(Xv)

    mejor, estado, t0 = -1, None, time.perf_counter()
    for ep in range(TR_EPOCHS):
        modelo_C.train()
        for ids, mask, lab in dl_tr:
            opt.zero_grad()
            loss = perdida(modelo_C(input_ids=ids.to(dev), attention_mask=mask.to(dev)).logits, lab.to(dev))
            loss.backward(); torch.nn.utils.clip_grad_norm_(modelo_C.parameters(), 1.0); opt.step()
        f1v = f1_score(yv, np.array(CLASES)[probas(modelo_C, dl_va).argmax(1)], average="macro")
        print(f"época {ep + 1}: pérdida {loss.item():.3f} · macro-F1 validación {f1v:.3f}")
        if f1v > mejor:
            mejor, estado = f1v, {k: v.detach().clone() for k, v in modelo_C.state_dict().items()}
    modelo_C.load_state_dict(estado); t_train = time.perf_counter() - t0

    t0 = time.perf_counter(); proba_C = probas(modelo_C, cargar(X_te)); t_inf = (time.perf_counter() - t0) / len(X_te) * 1000
    mb = sum(p.numel() for p in modelo_C.parameters()) * 4 / 1e6
    RESULTADOS["C · Transformer"] = {"proba": proba_C, "clases": CLASES, "t_train": t_train, "t_inf_ms": t_inf, "mb": mb,
                                     "cv": None, "modelo": modelo_C}
    print(f"C · Transformer: prueba macro-F1 {f1_score(y_te, np.array(CLASES)[proba_C.argmax(1)], average='macro'):.3f} · dispositivo {dev}")

## 4. Fase 4 · Evaluación: comparación justa y compuerta de calidad
Todos los modelos se evalúan en **la misma partición de prueba**:
- **Desempeño:** macro-F1 con intervalo de confianza por bootstrap y recall por clase (sobre todo el negativo).
- **Confianza:** calibración (ECE).
- **Costo:** tiempo de entrenamiento, latencia por reseña y tamaño del modelo. En un despliegue real, el costo también decide.

In [ ]:
def pred_de(r): return np.asarray(r["clases"])[r["proba"].argmax(1)]

def ece(y, r, bins=10):
    conf = r["proba"].max(1); ok = (pred_de(r) == np.asarray(y)).astype(float); e = 0.0
    for lo, hi in zip(np.linspace(0, 1, bins + 1)[:-1], np.linspace(0, 1, bins + 1)[1:]):
        m = (conf > lo) & (conf <= hi)
        if m.any(): e += m.mean() * abs(ok[m].mean() - conf[m].mean())
    return e

def bootstrap(y, p, n=1000):
    y, p = np.asarray(y), np.asarray(p); r = np.random.default_rng(SEED)
    v = [f1_score(y[i], p[i], average="macro") for i in (r.integers(0, len(y), len(y)) for _ in range(n))]
    return np.quantile(v, [0.025, 0.975])

filas = []
for nombre, r in RESULTADOS.items():
    p = pred_de(r); lo, hi = bootstrap(y_te, p)
    rec = recall_score(y_te, p, labels=CLASES, average=None, zero_division=0)
    filas.append({"modelo": nombre, "macro-F1": f1_score(y_te, p, average="macro"), "IC95 bajo": lo, "IC95 alto": hi,
                  **{f"recall {c}": v for c, v in zip(CLASES, rec)}, "exactitud": accuracy_score(y_te, p),
                  "ECE": ece(y_te, r), "CV macro-F1": r["cv"][0] if r["cv"] else np.nan,
                  "entrenar (s)": r["t_train"], "ms/reseña": r["t_inf_ms"], "tamaño (MB)": r["mb"]})
TABLA = pd.DataFrame(filas).set_index("modelo")
TABLA.round(3)

In [ ]:
modelos_eval = [n for n in RESULTADOS if not n.startswith("Dummy")]
fig, axes = plt.subplots(1, len(modelos_eval), figsize=(3.6 * len(modelos_eval), 3.4))
for ax, n in zip(np.atleast_1d(axes), modelos_eval):
    ConfusionMatrixDisplay.from_predictions(y_te, pred_de(RESULTADOS[n]), labels=CLASES, cmap="Reds", ax=ax, colorbar=False)
    ax.set_title(n, fontsize=8); ax.tick_params(labelsize=7)
plt.tight_layout(); plt.show()

### 4.1 ¿Las diferencias son reales? Prueba de McNemar entre pares
La prueba compara los aciertos de dos modelos **sobre las mismas reseñas** (Dietterich, 1998). Si p ≥ 0,05, no hay evidencia de que uno sea mejor que el otro; en ese caso conviene preferir el más barato.

In [ ]:
def mcnemar(a, b):
    ya = np.asarray(y_te); ok_a, ok_b = pred_de(RESULTADOS[a]) == ya, pred_de(RESULTADOS[b]) == ya
    sa, sb = int((ok_a & ~ok_b).sum()), int((~ok_a & ok_b).sum())
    return sa, sb, (binomtest(sa, sa + sb, 0.5).pvalue if sa + sb else 1.0)

pares = [(a, b) for i, a in enumerate(modelos_eval) for b in modelos_eval[i + 1:]]
pd.DataFrame([{"A": a, "B": b, "solo A acierta": x, "solo B acierta": y, "p": round(p, 4),
               "¿diferencia significativa?": "sí" if p < 0.05 else "no"} for a, b in pares for x, y, p in [mcnemar(a, b)]])

### 4.2 Calibración y umbral de automatización τ

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(11, 3.8))
ax[0].plot([0, 1], [0, 1], "--", color=GRIS)
for n, col in zip(modelos_eval, [GRIS, OSCURO, AZUL, "#7A9E3B", ROJO, "#8B2738"]):
    r = RESULTADOS[n]; conf = r["proba"].max(1); ok = pred_de(r) == np.asarray(y_te)
    b = np.linspace(0, 1, 11); idx = np.digitize(conf, b[1:-1])
    xs = [conf[idx == i].mean() for i in range(10) if (idx == i).any()]; ys = [ok[idx == i].mean() for i in range(10) if (idx == i).any()]
    ax[0].plot(xs, ys, "o-", color=col, label=n, ms=3)
    taus = np.arange(0.3, 0.96, 0.05)
    ax[1].plot(taus, [(conf >= t).mean() for t in taus], "-", color=col, label=n)
ax[0].set(xlabel="confianza", ylabel="exactitud real", title="Curvas de fiabilidad"); ax[0].legend(fontsize=6)
ax[1].axvline(TAU, ls="--", color=GRIS); ax[1].set(xlabel="τ", ylabel="% automatizado", title="Cobertura según τ")
plt.tight_layout(); plt.show()

### 4.3 Compuerta de calidad y selección del modelo

In [ ]:
dummy_f1 = TABLA.loc["Dummy (mayoritaria)", "macro-F1"]
neg = next((c for c in CLASES if normalizar(c).startswith("neg")), CLASES[0])
gate = pd.DataFrame({
    "macro-F1 ≥ mínimo": TABLA["macro-F1"] >= COMPUERTA["min_macro_f1"],
    "supera dummy": TABLA["macro-F1"] - dummy_f1 >= COMPUERTA["beats_dummy_by"],
    f"recall {neg} ≥ mínimo": TABLA[f"recall {neg}"] >= COMPUERTA["min_recall_negativo"],
}).drop(index="Dummy (mayoritaria)")
gate["APRUEBA"] = gate.all(axis=1)
display(gate.replace({True: "✔", False: "✘"}))

# Candidatos a desplegar: los tres modelos principales (A0 y B0 son ablaciones, solo para análisis)
CANDIDATOS = [n for n in gate.index if n.split(" ·")[0] in ("A", "B", "C")]
aprobados = TABLA.loc[[n for n in CANDIDATOS if gate.loc[n, "APRUEBA"]]].sort_values("macro-F1", ascending=False)
if aprobados.empty:
    ELEGIDO = None
    print("Ningún modelo pasa la compuerta: no se despliega. Revise datos, modelos o criterios.")
else:
    ELEGIDO = aprobados.index[0]
    # Regla de costo: si el mejor no es significativamente mejor que uno más barato que también aprobó, usar el más barato
    for alt in aprobados.index[1:]:
        if aprobados.loc[alt, "ms/reseña"] < aprobados.loc[ELEGIDO, "ms/reseña"]:
            _, _, p = mcnemar(ELEGIDO, alt)
            if p >= 0.05:
                print(f"{ELEGIDO} no es significativamente mejor que {alt} (p = {p:.3f}) y {alt} es más barato.")
                ELEGIDO = alt
                break
    print(f"Modelo elegido para desplegar: {ELEGIDO}")

### 4.4 Análisis de errores
Reseñas que **todos** los modelos fallan: suelen revelar ironía, reseñas mixtas, errores de etiquetado o falta de datos de esa clase.

In [ ]:
todos_fallan = np.ones(len(y_te), bool)
for n in modelos_eval: todos_fallan &= pred_de(RESULTADOS[n]) != np.asarray(y_te)
err = pd.DataFrame({"texto": X_te.values, "real": y_te.values,
                    **{n.split(" ·")[0]: pred_de(RESULTADOS[n]) for n in modelos_eval}})[todos_fallan]
print(f"{todos_fallan.sum()} reseñas fallan en todos los modelos")
err.head(10)

## 5. Fase 5 · Despliegue
Se exporta el modelo elegido con todo lo necesario para servirlo:
- el **modelo**;
- la **lógica de negocio**: aspectos, respuesta sugerida, guardarraíl y umbral τ;
- una **API**;
- una **ficha del modelo**.

Después se muestran tres formas de publicarlo.

In [ ]:
DEP = Path(DEPLOY_DIR); DEP.mkdir(exist_ok=True)
assert ELEGIDO is not None, "No hay modelo aprobado para desplegar"
es_transformer = ELEGIDO.startswith("C")

if es_transformer:
    (DEP / "modelo").mkdir(exist_ok=True)
    modelo_C.save_pretrained(DEP / "modelo"); tok.save_pretrained(DEP / "modelo")
    tipo = "transformer"
else:
    final = clone(RESULTADOS[ELEGIDO]["modelo"]).fit(d["texto"], d["etiqueta"])   # reentrenar con todos los datos
    joblib.dump(final, DEP / "modelo.joblib")
    tipo = "sklearn"

fila = TABLA.loc[ELEGIDO]
FICHA = {"modelo": ELEGIDO, "tipo": tipo, "clases": CLASES, "tau": TAU,
         "macro_f1_prueba": round(float(fila["macro-F1"]), 4), "ic95": [round(float(fila["IC95 bajo"]), 4), round(float(fila["IC95 alto"]), 4)],
         "ece": round(float(fila["ECE"]), 4), "compuerta": COMPUERTA, "datos": calidad,
         "uso_previsto": "Apoyo a un equipo humano que clasifica y responde reseñas; las respuestas son sugerencias.",
         "limitaciones": "Pocas reseñas negativas; ironía, negación y reseñas mixtas; posible deriva de los datos."}
(DEP / "ficha_modelo.json").write_text(json.dumps(FICHA, ensure_ascii=False, indent=2), encoding="utf-8")
print("Exportado en", DEP.resolve(), "→", sorted(p.name for p in DEP.iterdir()))

### 5.1 Servicio: lógica de negocio + API (FastAPI)
El archivo `deploy/servicio.py` se genera desde aquí. Contiene todo lo necesario para que el servicio funcione **fuera** del notebook.

In [ ]:
SERVICIO = r"""
# Traveler Insights · servicio de predicción (generado por el notebook)
import json, re, unicodedata
from pathlib import Path
from fastapi import FastAPI
from pydantic import BaseModel, Field

BASE = Path(__file__).parent
FICHA = json.loads((BASE / "ficha_modelo.json").read_text(encoding="utf-8"))

def normalizar(t):
    t = unicodedata.normalize("NFKD", str(t).lower())
    return re.sub(r"\s+", " ", "".join(c for c in t if not unicodedata.combining(c))).strip()

ASPECTOS = {"limpieza": r"limpi|suci|aseo|olor|bano", "servicio": r"servicio|atenci|personal|amabl|grosero|guia",
            "precio": r"precio|caro|barat|costo|cobr|tarifa", "ubicacion": r"ubicaci|cerca|lejos|acceso|transporte",
            "comida": r"comida|desayun|almuerz|cena|restaurante|plato|sabor", "instalaciones": r"habitaci|cama|piscina|wifi|ruido|ducha",
            "experiencia": r"paisaje|tour|recorrido|experiencia|visita|playa|buceo"}
COMPROMISOS = re.compile(r"reembols|devoluci|descuento|compensa|gratis|garantiz")

def aspectos(t): return [a for a, p in ASPECTOS.items() if re.search(p, normalizar(t))]

def respuesta(t, clase):
    a = aspectos(t); menc = f" sobre {', '.join(a)}" if a else ""
    c = normalizar(clase)
    if c.startswith("pos"): return f"¡Gracias por tu reseña{menc}! Te esperamos de nuevo."
    if c.startswith("neg"): return f"Lamentamos tu experiencia{menc}. Un integrante de nuestro equipo se comunicará contigo."
    return f"Gracias por tus comentarios{menc}. Nos ayudan a mejorar."

if FICHA["tipo"] == "sklearn":
    import sys, joblib
    sys.path.insert(0, str(BASE))
    import ti_rasgos  # noqa: F401  (clases y funciones que el .joblib necesita)
    MODELO = joblib.load(BASE / "modelo.joblib")
    def predecir(textos):
        P = MODELO.predict_proba(textos); return [dict(zip(map(str, MODELO.classes_), map(float, p))) for p in P]
else:
    import torch
    from transformers import AutoModelForSequenceClassification, AutoTokenizer
    TOK = AutoTokenizer.from_pretrained(BASE / "modelo")
    MODELO = AutoModelForSequenceClassification.from_pretrained(BASE / "modelo").eval()
    def predecir(textos):
        with torch.no_grad():
            enc = TOK(textos, truncation=True, max_length=128, padding=True, return_tensors="pt")
            P = torch.softmax(MODELO(**enc).logits, -1).numpy()
        return [dict(zip(FICHA["clases"], map(float, p))) for p in P]

def analizar(textos, tau=None):
    tau = FICHA["tau"] if tau is None else tau
    out = []
    for t, pr in zip(textos, predecir(textos)):
        clase = max(pr, key=pr.get); conf = pr[clase]; r = respuesta(t, clase)
        motivos = (["confianza baja"] if conf < tau else []) + (["respuesta con compromisos"] if COMPROMISOS.search(normalizar(r)) else [])
        out.append({"texto": t, "clase": clase, "confianza": round(conf, 4), "aspectos": aspectos(t),
                    "respuesta_sugerida": r, "decision": "revision_humana" if motivos else "automatizar", "motivos": motivos})
    return out

app = FastAPI(title="Traveler Insights")

class Peticion(BaseModel):
    texts: list[str] = Field(..., min_length=1, max_length=200)
    tau: float | None = Field(None, ge=0, le=1)

@app.get("/api/health")
def salud(): return {"status": "ok", "modelo": FICHA["modelo"], "macro_f1": FICHA["macro_f1_prueba"]}

@app.post("/api/analyze")
def analyze(p: Peticion): return {"results": analizar(p.texts, p.tau)}
"""
(DEP / "servicio.py").write_text(SERVICIO.lstrip(), encoding="utf-8")

In [ ]:
# El .joblib referencia clases y funciones de ti_rasgos.py: el módulo viaja con el modelo
import shutil
shutil.copy("ti_rasgos.py", DEP / "ti_rasgos.py")
(DEP / "requirements.txt").write_text(
    "fastapi\nuvicorn\n" + ("torch\ntransformers\n" if es_transformer else "scikit-learn\nscipy\nnumpy\njoblib\n" +
                              ("gensim\n" if ELEGIDO.startswith("B") else "")), encoding="utf-8")
print((DEP / "requirements.txt").read_text())

**Prueba del servicio sin levantar un servidor** (exactamente el código que se desplegará):

In [ ]:
import importlib, sys
sys.path.insert(0, str(DEP.resolve()))
sys.modules.pop("servicio", None)
servicio = importlib.import_module("servicio")
from fastapi.testclient import TestClient
cli = TestClient(servicio.app)
print(cli.get("/api/health").json())
r = cli.post("/api/analyze", json={"texts": list(X_te.head(3))}).json()["results"]
pd.DataFrame(r)[["texto", "clase", "confianza", "aspectos", "decision", "motivos"]]

### 5.2 Opciones para publicarlo

| Opción | Cuándo usarla | Pasos |
|---|---|---|
| **Local / servidor propio** | Pruebas o red interna | `cd deploy && pip install -r requirements.txt && uvicorn servicio:app` → http://127.0.0.1:8000/docs |
| **Hugging Face Spaces** (Docker o Gradio) | Modelo C (Transformer) o cualquier modelo; gratis con CPU | Crear un *Space*, subir la carpeta `deploy/` y un `Dockerfile` que ejecute `uvicorn servicio:app --host 0.0.0.0 --port 7860` |
| **Vercel** (función serverless) | Solo modelos livianos (A); el paquete debe pesar menos de ~250 MB | scikit-learn y scipy se acercan al límite. Lo robusto es exportar el modelo A a JSON e inferir en Python puro, como en el repositorio *TravelerInsights* |

El Transformer pesa cientos de MB y requiere `torch`, así que **no cabe en Vercel**: use Hugging Face Spaces, un servidor con GPU o un endpoint de inferencia.

In [ ]:
(DEP / "Dockerfile").write_text(
    "FROM python:3.12-slim\nWORKDIR /app\nCOPY . .\nRUN pip install --no-cache-dir -r requirements.txt\n"
    "EXPOSE 7860\nCMD [\"uvicorn\", \"servicio:app\", \"--host\", \"0.0.0.0\", \"--port\", \"7860\"]\n", encoding="utf-8")
(DEP / "README.md").write_text(
    "---\ntitle: Traveler Insights\nsdk: docker\napp_port: 7860\n---\n\n"
    f"# Traveler Insights · {ELEGIDO}\n\nmacro-F1 en prueba: {FICHA['macro_f1_prueba']} (IC 95 % {FICHA['ic95']}).\n"
    "Endpoints: `GET /api/health`, `POST /api/analyze` con `{\"texts\": [\"...\"]}`. Las respuestas son sugerencias.\n", encoding="utf-8")
print("Carpeta lista para Hugging Face Spaces (sdk: docker):", sorted(p.name for p in DEP.iterdir()))

## 6. Fase 6 · Monitoreo y mantenimiento
En producción no hay etiquetas, pero sí se puede vigilar la **deriva**: si la distribución de clases predichas o de confianza cambia mucho respecto a la prueba, el modelo puede estar fuera de su zona de validez. `test.csv` hace el papel de "datos nuevos".

Se usa el índice de estabilidad poblacional (PSI). Como regla práctica: < 0,1 estable; entre 0,1 y 0,25, cambio moderado; > 0,25, revisar.

In [ ]:
def psi(a, b, eps=1e-6):
    a, b = np.asarray(a, float) + eps, np.asarray(b, float) + eps; a, b = a / a.sum(), b / b.sum()
    return float(np.sum((b - a) * np.log(b / a)))

if Path(TEST_PATH).exists():
    nuevos = pd.read_csv(TEST_PATH)[TEXT_COL].fillna("").astype(str).tolist()
    ref = servicio.analizar(list(X_te)); new = servicio.analizar(nuevos)
    cr = pd.Series([x["clase"] for x in ref]).value_counts().reindex(CLASES, fill_value=0)
    cn = pd.Series([x["clase"] for x in new]).value_counts().reindex(CLASES, fill_value=0)
    b = np.linspace(0, 1, 11)
    hr = np.histogram([x["confianza"] for x in ref], b)[0]; hn = np.histogram([x["confianza"] for x in new], b)[0]
    print(f"PSI de clases: {psi(cr, cn):.3f} · PSI de confianza: {psi(hr, hn):.3f}")
    print(f"% a revisión humana → prueba: {np.mean([x['decision'] != 'automatizar' for x in ref]):.0%} · nuevos: {np.mean([x['decision'] != 'automatizar' for x in new]):.0%}")
    display(pd.DataFrame({"prueba": cr / cr.sum(), "nuevos (test.csv)": cn / cn.sum()}).round(3))
else:
    print("No se encontró test.csv: descárguelo de Kaggle para esta sección.")

**Plan de mantenimiento:**
- **Auditoría semanal por muestreo** de lo automatizado.
- **Tasa de corrección humana:** cuántas clases o respuestas modifica el equipo.
- **PSI mensual.**
- **Reentrenar** cuando haya nuevas reseñas etiquetadas, sobre todo negativas, o cuando el PSI supere 0,25.
- **Toda versión nueva pasa por la misma compuerta de §4.3.**

## 7. Reflexión crítica (entregable)
1. ¿Cuánto aporta el léxico en A y en B? ¿La diferencia es significativa?
2. ¿El Transformer supera **significativamente** a A? ¿Justifica su costo de latencia, tamaño e infraestructura?
3. ¿Qué modelo detecta mejor las quejas (recall negativo)? ¿Por qué es la métrica más importante para el negocio?
4. ¿Qué representación conserva la negación? Pruebe "no estuvo nada mal" en los tres modelos.
5. ¿Qué τ recomendaría para cada modelo, según su calibración?
6. ¿Qué cambiaría en la compuerta de calidad después de ver estos resultados? ¿Por qué no debería cambiarla para *esta* versión?

### Referencias
- Studer, S. et al. (2021). Towards CRISP-ML(Q). *Machine Learning and Knowledge Extraction*, 3(2), 392–413.
- Jurafsky, D. y Martin, J. H. *Speech and Language Processing* (3.ª ed., borrador): caps. 4 (clasificación), 6 (embeddings) y 11 (modelos preentrenados y afinamiento).
- Bojanowski, P. et al. (2017). Enriching Word Vectors with Subword Information. *TACL*, 5, 135–146.
- Cañete, J. et al. (2020). Spanish Pre-trained BERT Model and Evaluation Data. *PML4DC @ ICLR 2020*.
- Dietterich, T. G. (1998). Approximate Statistical Tests for Comparing Supervised Classification Learning Algorithms. *Neural Computation*, 10(7).
- Guo, C. et al. (2017). On Calibration of Modern Neural Networks. *ICML 2017*.